# Assignment 2: Evaluation Notebook
Loads the trained checkpoint and tokenizer, translates the provided test set, writes submission.csv and reports BLEU, BERTScore, inference time and parameter count.

Upload `best_model.pt`, `spm_joint.model` and the test CSV files to /content/ before running. If BERTScore raises a transformers version error, restart the runtime once after the install cell and run again.

In [1]:
!pip install -q sentencepiece nltk bert-score "transformers==4.46.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [8]:
import pandas as pd
import torch
import torch.nn as nn
import math, time
import sentencepiece as spm
from nltk.translate.bleu_score import corpus_bleu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

TEST_SA = "/content/test_sa_1000.csv"
TEST_EN = "/content/test_en_1000.csv"


cuda


In [9]:
sp = spm.SentencePieceProcessor(model_file="spm_joint.model")
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
MAX_LEN = 64


In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4,
                 dim_ff=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_ff, dropout=dropout, batch_first=True,
        )
        self.out = nn.Linear(d_model, vocab_size)
        self.out.weight = self.embed.weight  # weight tying

    def forward(self, src, tgt_in):
        src_pad = src == PAD_ID
        tgt_pad = tgt_in == PAD_ID
        causal = nn.Transformer.generate_square_subsequent_mask(tgt_in.size(1)).to(src.device)
        src_emb = self.pos_enc(self.embed(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.embed(tgt_in) * math.sqrt(self.d_model))
        h = self.transformer(src_emb, tgt_emb, tgt_mask=causal,
                             src_key_padding_mask=src_pad,
                             tgt_key_padding_mask=tgt_pad,
                             memory_key_padding_mask=src_pad)
        return self.out(h)


In [11]:
model = Seq2SeqTransformer(sp.get_piece_size()).to(device)
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")


Parameters: 9,429,824


In [12]:
@torch.no_grad()
def translate_greedy(sentences, batch_size=64, max_len=MAX_LEN):
    results = []
    for i in range(0, len(sentences), batch_size):
        chunk = [str(s).strip() for s in sentences[i:i + batch_size]]
        enc = [sp.encode(s)[:max_len - 1] + [EOS_ID] for s in chunk]
        src_max = max(len(e) for e in enc)
        src = torch.full((len(enc), src_max), PAD_ID, dtype=torch.long, device=device)
        for j, e in enumerate(enc):
            src[j, :len(e)] = torch.tensor(e, device=device)
        src_pad = src == PAD_ID
        memory = model.transformer.encoder(
            model.pos_enc(model.embed(src) * math.sqrt(model.d_model)),
            src_key_padding_mask=src_pad,
        )
        tgt = torch.full((len(enc), 1), BOS_ID, dtype=torch.long, device=device)
        done = torch.zeros(len(enc), dtype=torch.bool, device=device)
        for _ in range(max_len):
            causal = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(device)
            h = model.transformer.decoder(
                model.pos_enc(model.embed(tgt) * math.sqrt(model.d_model)),
                memory, tgt_mask=causal, memory_key_padding_mask=src_pad,
            )
            nxt = model.out(h[:, -1]).argmax(-1, keepdim=True)
            nxt[done] = PAD_ID
            tgt = torch.cat([tgt, nxt], dim=1)
            done |= nxt.squeeze(1) == EOS_ID
            if done.all():
                break
        for row in tgt.tolist():
            toks = [t for t in row if t not in (BOS_ID, EOS_ID, PAD_ID)]
            results.append(sp.decode(toks))
    return results

In [13]:
test_sa_df = pd.read_csv(TEST_SA)

t0 = time.time()
preds = translate_greedy(test_sa_df["Sentence_sa"].tolist())
inference_time = time.time() - t0

submission = pd.DataFrame({"Source_id": test_sa_df["Source_id"], "Sentence_en": preds})
submission.to_csv("submission.csv", index=False, encoding="utf-8")
print(f"Inference time : {inference_time:.2f}s")
submission.head()


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Inference time : 13.75s


,Source_id,Sentence_en
0,1,And the law of the i.
1,2,"""And I say unto you, that I say unto you, that..."
2,3,I have a new page to click on the I will click...
3,4,"So, let's 1stmana by 1stonana by 1st."
4,5,I have already in the pain of the I have a two...


In [14]:
test_en_df = pd.read_csv(TEST_EN)

refs = [[str(r).split()] for r in test_en_df["Sentence_en"]]
hyps = [p.split() for p in preds]
bleu = corpus_bleu(refs, hyps)

from bert_score import score as bert_score
P, R, F1 = bert_score(preds, test_en_df["Sentence_en"].astype(str).tolist(),
                      lang="en", rescale_with_baseline=True)

print(f"BLEU           : {bleu:.4f}")
print(f"BERTScore F1   : {F1.mean().item():.4f}")
print(f"Inference time : {inference_time:.2f}s")
print(f"Parameters     : {n_params:,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BLEU           : 0.0314
BERTScore F1   : 0.1036
Inference time : 13.75s
Parameters     : 9,429,824
